[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/06_pandas_intro/06_3_Selecting_Filtering.ipynb)

# 06.3: Selecting, Filtering, and Slicing

The Titanic dataset has 887 rows. Almost no real analysis works on all rows at once. You'll constantly want to zoom in on a specific group: passengers who survived, first-class passengers older than 40, or a single record you want to check. This notebook gives you the complete toolkit for extracting exactly the rows and columns you need.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

,survived,pclass,name,sex,age,sibsp,parch,fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


## The Problem This Notebook Solves

The Titanic dataset has 887 rows. Almost no real analysis works on all rows at once.

You'll constantly want to zoom in on a specific group:
- *"Show me only the passengers who survived."*
- *"Show me first-class passengers older than 40."*
- *"Show me the record for this specific passenger."*
- *"Show me just the name, age, and survival columns."*

This notebook gives you the complete toolkit for extracting exactly the subset of rows and columns you need.

## 1. Looking Up a Specific Row

Sometimes you want a single record — maybe to spot-check a value or investigate an outlier. Two tools do this:

- **`.loc[label]`** — selects by the index label (here, the row number).
- **`.iloc[position]`** — selects by integer position, 0-based, like a Python list.

When the index is just 0, 1, 2, … (as it is here), both give the same result. The difference matters when you've filtered or sorted the DataFrame and the index no longer matches position.

In [2]:
# Look up a single row by its label
df.loc[0]

survived                         0
pclass                           3
name        Mr. Owen Harris Braund
sex                           male
age                           22.0
sibsp                            1
parch                            0
fare                          7.25
Name: 0, dtype: object

In [3]:
# Look up specific columns for a specific row
df.loc[0, ["name", "age", "pclass", "survived"]]

name        Mr. Owen Harris Braund
age                           22.0
pclass                           3
survived                         0
Name: 0, dtype: object

In [4]:
# .iloc[] uses position, not label
# After filtering, row 0 may not be at position 0 — iloc always gives the nth row
print("First row via iloc:", df.iloc[0]["name"])
print("Last row via iloc: ", df.iloc[-1]["name"])

First row via iloc: Mr. Owen Harris Braund
Last row via iloc:  Mr. Patrick Dooley


### Selecting a Range of Rows

Both `.loc[]` and `.iloc[]` support slicing, but with one important difference:
- `.loc[]` slice end is **inclusive** (the row with that label is included).
- `.iloc[]` slice end is **exclusive** (like standard Python slicing).

In [5]:
# .iloc[]: positions 0 through 4 — gives 5 rows (end exclusive)
df.iloc[0:5][["name", "age", "survived"]]

,name,age,survived
0,Mr. Owen Harris Braund,22.0,0
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,38.0,1
2,Miss. Laina Heikkinen,26.0,1
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,35.0,1
4,Mr. William Henry Allen,35.0,0


In [6]:
# Selecting specific rows AND specific columns simultaneously
df.loc[[0, 5, 10], ["name", "pclass", "survived"]]

,name,pclass,survived
0,Mr. Owen Harris Braund,3,0
5,Mr. James Moran,3,0
10,Miss. Marguerite Rut Sandstrom,3,1


## 2. Filtering Rows: Boolean Masks

Single-row lookups are useful for spot-checking, but most analyses focus on a group defined by a condition. To do that, you write a comparison expression on a column. The result is a **boolean Series** — a column of `True`/`False` values telling you which rows match.

Then you pass that boolean Series back into `.loc[]`, and pandas keeps only the `True` rows.

In [7]:
# Who survived?
survived_mask = df["survived"] == 1
# This is 887 True/False values — one per passenger
survived_mask.head(10)

0    False
1     True
2     True
3     True
4    False
5    False
6    False
7    False
8     True
9     True
Name: survived, dtype: bool

In [8]:
survivors = df.loc[survived_mask]
print(f"{len(survivors)} of {len(df)} passengers survived ({len(survivors)/len(df):.0%})")
survivors[["name", "sex", "pclass", "age", "fare"]].head()

342 of 887 passengers survived (39%)


,name,sex,pclass,age,fare
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,1,38.0,71.2833
2,Miss. Laina Heikkinen,female,3,26.0,7.9250
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,1,35.0,53.1000
8,Mrs. Oscar W (Elisabeth Vilhelmina Berg) Johnson,female,3,27.0,11.1333
9,Mrs. Nicholas (Adele Achem) Nasser,female,2,14.0,30.0708


In [9]:
# You'll almost always write the mask inline, without naming it separately
first_class = df.loc[df["pclass"] == 1]
print(f"First-class passengers: {len(first_class)}")
first_class[["name", "pclass", "fare"]].head()

First-class passengers: 216


,name,pclass,fare
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,1,71.2833
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,1,53.1000
6,Mr. Timothy J McCarthy,1,51.8625
11,Miss. Elizabeth Bonnell,1,26.5500
23,Mr. William Thompson Sloper,1,35.5000


## 3. Combining Conditions

Real questions usually involve more than one condition. Use `&` (AND) and `|` (OR) to combine boolean masks.

**Always put each condition in its own parentheses.** Python's operator precedence evaluates `&` before `==`, so without parentheses the expression silently does the wrong thing.

In [10]:
# First-class passengers who survived
first_class_survivors = df.loc[
    (df["pclass"] == 1) & (df["survived"] == 1)
]
print(f"First-class survivors: {len(first_class_survivors)}")
first_class_survivors[["name", "sex", "age", "fare"]].head()

First-class survivors: 136


,name,sex,age,fare
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,71.2833
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,53.1000
11,Miss. Elizabeth Bonnell,female,58.0,26.5500
23,Mr. William Thompson Sloper,male,28.0,35.5000
31,Mrs. William Augustus (Marie Eugenie) Spencer,female,48.0,146.5208


In [11]:
# Women who were in first OR second class
upper_class_women = df.loc[
    (df["sex"] == "female") & (df["pclass"].isin([1, 2]))
]
print(f"Women in 1st or 2nd class: {len(upper_class_women)}")
print(f"Of those, survived: {upper_class_women['survived'].sum()}")

Women in 1st or 2nd class: 170
Of those, survived: 161


### `.isin()` — Clean Multi-Value Matching

When you want to check whether a value belongs to a set of options, `.isin([...])` is cleaner than chaining multiple `|` conditions.

In [12]:
# Equivalent to (pclass == 1) | (pclass == 2), but easier to read
df.loc[df["pclass"].isin([1, 2])][["name", "pclass"]].head()

,name,pclass
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,1
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,1
6,Mr. Timothy J McCarthy,1
9,Mrs. Nicholas (Adele Achem) Nasser,2
11,Miss. Elizabeth Bonnell,1


### `~` — Negation (NOT)

Prefix a mask with `~` to invert it — keep only the rows where the condition is `False`.

In [13]:
# Passengers who did NOT survive
did_not_survive = df.loc[~(df["survived"] == 1)]
# Equivalently: df.loc[df["survived"] == 0]
print(f"Did not survive: {len(did_not_survive)}")

Did not survive: 545


## 4. `.query()` — More Readable Filters

For simple conditions, `.query()` accepts a plain-English string and often reads more naturally than a boolean mask.

In [14]:
# Same as df.loc[(df["pclass"] == 1) & (df["age"] > 40)]
older_first_class = df.query("pclass == 1 and age > 40")
older_first_class[["name", "pclass", "age", "fare"]].head()

,name,pclass,age,fare
6,Mr. Timothy J McCarthy,1,54.0,51.8625
11,Miss. Elizabeth Bonnell,1,58.0,26.5500
31,Mrs. William Augustus (Marie Eugenie) Spencer,1,48.0,146.5208
35,Mr. Alexander Oskar Holverson,1,42.0,52.0000
51,Mrs. Henry Sleeper (Myna Haxtun) Harper,1,49.0,76.7292


In [15]:
# 'in' works naturally inside .query()
df.query("pclass in [1, 2] and survived == 1")[["name", "pclass"]].head()

,name,pclass
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,1
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,1
9,Mrs. Nicholas (Adele Achem) Nasser,2
11,Miss. Elizabeth Bonnell,1
15,Mrs. (Mary D Kingcome) Hewlett,2


`.query()` has limits: it doesn't work well with column names containing spaces or special characters, and it can't reference complex computed values. Use boolean masks when you need full flexibility — but reach for `.query()` when the condition is simple and you want it to be easy to read.

## 5. Copy vs View — A Practical Warning

When you filter a DataFrame, pandas sometimes returns a **view** (a live window into the original data) and sometimes a **copy** (completely independent). The distinction matters the moment you try to *modify* the result.

If you modify a view, you may accidentally modify the original. If you modify a copy, your change may silently disappear. pandas warns you about this ambiguity with `SettingWithCopyWarning`.

In [16]:
# This may raise SettingWithCopyWarning
survivors = df[df["survived"] == 1]
survivors["fare"] = 0   # Are we changing a copy or the original?

In [17]:
# The fix: call .copy() immediately after any filter you plan to modify
survivors = df[df["survived"] == 1].copy()
survivors["fare"] = 0   # Clearly modifying an independent copy — no warning

# Confirm the original is unchanged
print("Original fare for row 1:", df.loc[1, "fare"])

Original fare for row 1: 71.2833


**Rule:** if you filter and plan to *read* the result, `.copy()` is unnecessary. If you filter and plan to *add or change columns*, always call `.copy()` first.

## 6. Putting It Together: Answering a Real Question

Let's put these tools to work on a question that requires combining several of them:

*"Among third-class women, who survived and who didn't — and what were they paying?"*


In [18]:
third_class_women = df.loc[
    (df["pclass"] == 3) & (df["sex"] == "female")
].copy()

survived = third_class_women.loc[third_class_women["survived"] == 1]
did_not  = third_class_women.loc[third_class_women["survived"] == 0]

print(f"Third-class women:         {len(third_class_women)}")
print(f"  Survived:                {len(survived)} ({len(survived)/len(third_class_women):.0%})")
print(f"  Did not survive:         {len(did_not)} ({len(did_not)/len(third_class_women):.0%})")
print()
print(f"Average fare — survivors:     £{survived['fare'].mean():.2f}")
print(f"Average fare — did not survive: £{did_not['fare'].mean():.2f}")

Third-class women:         144
  Survived:                72 (50%)
  Did not survive:         72 (50%)

Average fare — survivors:     £12.46
Average fare — did not survive: £19.77


## What's next

You can now extract any subset of rows and columns you need. But a filtered result is only trustworthy if the underlying data is clean — types correct, missing values handled, duplicates removed, strings consistent. Notebook 06.4 covers the most common data cleaning operations you will need before any analysis can be trusted.